# SIS3: Generative Model for Text Generation
## Part 1: Baseline LSTM Implementation

**Architecture:** Character-level LSTM  
**Dataset:** Shakespeare Text Corpus (tinyshakespeare)

## Cell 1 — Install & Import Libraries

In [1]:
# Uncomment and run this line ONCE if any library is missing
# !pip install torch numpy matplotlib requests jupyter

import os
import time
import math
import json
import random
import requests
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch version : {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {device}')

PyTorch version : 2.10.0+cpu
Using device    : cpu


## Cell 2 — Configuration

In [2]:
#  Baseline hyper-parameters (exactly as in SIS3 spec)
SEQ_LEN       = 100      # characters per input sequence
EMBED_DIM     = 256      # embedding dimension
LSTM_UNITS    = 512      # LSTM hidden units (both layers)
DROPOUT_RATE  = 0.2      # dropout after second LSTM
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
EPOCHS        = 30       # set to 10 for a quick test run
VAL_SPLIT     = 0.2      # 20 % held-out validation
STEP          = 3        # sliding window stride
SEED          = 42

#  Dataset 
DATA_URL   = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
LOCAL_PATH = None   # e.g. r'C:\Users\aitzh\Downloads\input.txt'  ← set if downloaded

#  Output folder 
OUT_DIR = 'sis3_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print('Configuration loaded ✓')

Configuration loaded ✓


## Cell 3 — Load Dataset

In [4]:
LOCAL_PATH = r"C:\Users\aitzh\OneDrive\Desktop\deep learinig\shakespeare.txt"

with open(LOCAL_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Corpus length: {len(text):,} characters')
print(text[:300])

Corpus length: 5,436,475 characters
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine whe


## Cell 4 — Build Vocabulary

In [5]:
chars      = sorted(set(text))
VOCAB_SIZE = len(chars)
char2idx   = {ch: i for i, ch in enumerate(chars)}
idx2char   = {i: ch for i, ch in enumerate(chars)}

print(f'Unique characters (vocab size): {VOCAB_SIZE}')
print(f'Characters: {"".join(chars)}')

Unique characters (vocab size): 84
Characters: 
 !"&'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_`abcdefghijklmnopqrstuvwxyz|}


## Cell 5 — PyTorch Dataset & DataLoader

In [6]:
class ShakespeareDataset(Dataset):
    """Sliding-window character dataset."""

    def __init__(self, encoded_text, seq_len, step):
        self.sequences  = []
        self.targets    = []
        for i in range(0, len(encoded_text) - seq_len, step):
            self.sequences.append(encoded_text[i : i + seq_len])
            self.targets.append(encoded_text[i + seq_len])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        x = torch.tensor(self.sequences[idx], dtype=torch.long)
        y = torch.tensor(self.targets[idx],   dtype=torch.long)
        return x, y


# Encode corpus
encoded = [char2idx[c] for c in text]

# Split indices 
n_chars  = len(encoded)
val_end  = int(n_chars * VAL_SPLIT)

# Keep temporal order: first 20 % → val, rest → train  (avoids data leakage)
val_encoded   = encoded[:val_end]
train_encoded = encoded[val_end:]

train_dataset = ShakespeareDataset(train_encoded, SEQ_LEN, STEP)
val_dataset   = ShakespeareDataset(val_encoded,   SEQ_LEN, STEP)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Training sequences  : {len(train_dataset):,}')
print(f'Validation sequences: {len(val_dataset):,}')

Training sequences  : 1,449,694
Validation sequences: 362,399


## Cell 6 — Build Baseline Model

In [7]:
class BaselineLSTM(nn.Module):
    """
    Baseline architecture (SIS3 spec):
      Input → Embedding(256) → LSTM(512) → LSTM(512) → Dropout(0.2)
            → Linear(vocab_size) → LogSoftmax
    """

    def __init__(self, vocab_size, embed_dim, lstm_units, dropout_rate):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm1     = nn.LSTM(embed_dim,   lstm_units, batch_first=True)
        self.lstm2     = nn.LSTM(lstm_units,  lstm_units, batch_first=True)
        self.dropout   = nn.Dropout(dropout_rate)
        self.fc        = nn.Linear(lstm_units, vocab_size)

    def forward(self, x):
        # x : (batch, seq_len)  — integer character indices
        out = self.embedding(x)              # (batch, seq_len, embed_dim)
        out, _ = self.lstm1(out)             # (batch, seq_len, lstm_units)
        out, _ = self.lstm2(out)             # (batch, seq_len, lstm_units)
        out = out[:, -1, :]                  # last timestep → (batch, lstm_units)
        out = self.dropout(out)
        out = self.fc(out)                   # (batch, vocab_size)  — raw logits
        return out


model = BaselineLSTM(
    vocab_size   = VOCAB_SIZE,
    embed_dim    = EMBED_DIM,
    lstm_units   = LSTM_UNITS,
    dropout_rate = DROPOUT_RATE
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

BaselineLSTM(
  (embedding): Embedding(84, 256)
  (lstm1): LSTM(256, 512, batch_first=True)
  (lstm2): LSTM(512, 512, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=512, out_features=84, bias=True)
)

Total trainable parameters: 3,742,804


## Cell 7 — Train Baseline Model

In [8]:
criterion   = nn.CrossEntropyLoss()               # expects raw logits
optimizer   = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop 
history = {'train_loss': [], 'val_loss': []}

best_val_loss    = float('inf')
best_model_path  = os.path.join(OUT_DIR, 'baseline_best.pt')
patience_counter = 0
PATIENCE         = 5        # early stopping patience

train_start = time.time()
print(f'Training on {device} ...\n')

for epoch in range(1, EPOCHS + 1):
    #  Train phase
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)          # (batch, vocab_size)
        loss   = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # gradient clipping
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

    train_loss = running_loss / len(train_dataset)

    #  Validation phase 
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            val_running_loss += loss.item() * X_batch.size(0)

    val_loss = val_running_loss / len(val_dataset)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f'Epoch {epoch:3d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val PPX: {math.exp(val_loss):.2f}')

    #  Save best & early stopping 
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        patience_counter = 0
        print(f'            Best model saved (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

total_train_time = time.time() - train_start

# Reload best weights 
model.load_state_dict(torch.load(best_model_path, map_location=device))
print(f'\nBest model reloaded ✓')
print(f'Total training time : {total_train_time:.1f} s  ({total_train_time/60:.1f} min)')

Training on cpu ...



c:\Users\aitzh\OneDrive\Desktop\deep learinig\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## Cell 8 — Calculate Performance Metrics

In [ ]:
final_train_loss = history['train_loss'][-1]
final_val_loss   = history['val_loss'][-1]
best_val_epoch   = int(np.argmin(history['val_loss'])) + 1

train_perplexity = math.exp(final_train_loss)
val_perplexity   = math.exp(final_val_loss)
best_perplexity  = math.exp(best_val_loss)

# Convergence: first epoch that reached 90% of total improvement
val_losses     = history['val_loss']
initial_loss   = val_losses[0]
target_loss    = initial_loss - 0.9 * (initial_loss - best_val_loss)
convergence_ep = next(
    (i + 1 for i, v in enumerate(val_losses) if v <= target_loss),
    len(val_losses)
)

print('=' * 57)
print('        BASELINE MODEL — PERFORMANCE METRICS')
print('=' * 57)
print(f'  Final Training Loss    : {final_train_loss:.4f}')
print(f'  Final Validation Loss  : {final_val_loss:.4f}')
print(f'  Best  Validation Loss  : {best_val_loss:.4f}  (epoch {best_val_epoch})')
print(f'  Training Perplexity    : {train_perplexity:.2f}')
print(f'  Validation Perplexity  : {val_perplexity:.2f}')
print(f'  Best Val. Perplexity   : {best_perplexity:.2f}')
print(f'  Total Training Time    : {total_train_time:.1f} s')
print(f'  Trainable Parameters   : {total_params:,}')
print(f'  Epochs Run             : {len(val_losses)}')
print(f'  Best Epoch             : {best_val_epoch}')
print(f'  Convergence Epoch      : {convergence_ep}')
print('=' * 57)

## Cell 9 — Plot Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline LSTM — Training Curves', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_ran, history['train_loss'], label='Training Loss',   color='steelblue',  linewidth=2)
axes[0].plot(epochs_ran, history['val_loss'],   label='Validation Loss', color='darkorange', linewidth=2, linestyle='--')
axes[0].axvline(best_val_epoch, color='green', linestyle=':', alpha=0.8, label=f'Best epoch ({best_val_epoch})')
axes[0].set_title('Loss (Cross-Entropy)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perplexity
train_ppx = [math.exp(v) for v in history['train_loss']]
val_ppx   = [math.exp(v) for v in history['val_loss']]

axes[1].plot(epochs_ran, train_ppx, label='Training Perplexity',   color='steelblue',  linewidth=2)
axes[1].plot(epochs_ran, val_ppx,   label='Validation Perplexity', color='darkorange', linewidth=2, linestyle='--')
axes[1].axvline(best_val_epoch, color='green', linestyle=':', alpha=0.8, label=f'Best epoch ({best_val_epoch})')
axes[1].set_title('Perplexity  (lower = better)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('exp(loss)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, 'baseline_training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {plot_path}')

## Cell 10 — Text Generation Function

In [ ]:
def sample_from_distribution(logits, temperature=1.0):
    """
    Sample next character index from model logits.
    temperature < 1  → conservative  |  temperature > 1  → creative
    """
    logits = logits / temperature
    probs  = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


def generate_text(model, seed_text, char2idx, idx2char,
                  gen_length=500, temperature=1.0):
    """
    Autoregressively generate text from a seed string.
    """
    model.eval()

    # Ensure seed is exactly SEQ_LEN chars
    if len(seed_text) < SEQ_LEN:
        seed_text = text[:SEQ_LEN - len(seed_text)] + seed_text
    seed_text = seed_text[-SEQ_LEN:]

    current_seq = [char2idx.get(c, 0) for c in seed_text]
    generated   = ''

    with torch.no_grad():
        for _ in range(gen_length):
            x     = torch.tensor([current_seq], dtype=torch.long).to(device)  # (1, SEQ_LEN)
            logits = model(x)[0]                                               # (vocab_size,)
            next_idx  = sample_from_distribution(logits, temperature)
            next_char = idx2char[next_idx]
            generated   += next_char
            current_seq  = current_seq[1:] + [next_idx]

    return generated


print('generate_text() ready ✓')

## Cell 11 — Generate Text at Different Temperatures

In [ ]:
start_idx = random.randint(0, len(text) - SEQ_LEN - 1)
seed      = text[start_idx : start_idx + SEQ_LEN]

print(f'Seed (first 80 chars): "{seed[:80]}..."\n')
print('=' * 60)

temperatures     = [0.2, 0.5, 1.0, 1.2]
generated_samples = {}

for temp in temperatures:
    print(f'\n── Temperature = {temp} ──────────────────────────────────')
    sample = generate_text(
        model, seed, char2idx, idx2char,
        gen_length=400, temperature=temp
    )
    generated_samples[temp] = sample
    print(sample)

## Cell 12 — Save Generated Samples

In [ ]:
samples_path = os.path.join(OUT_DIR, 'baseline_generated_samples.txt')

with open(samples_path, 'w', encoding='utf-8') as f:
    f.write('SIS3 — BASELINE LSTM GENERATED TEXT SAMPLES\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Seed (first 80 chars): {seed[:80]}...\n')
    f.write(f'Epochs run: {len(val_losses)} | Best Val Loss: {best_val_loss:.4f} | Best Val PPX: {best_perplexity:.2f}\n\n')
    for temp, sample in generated_samples.items():
        f.write(f'─── Temperature = {temp} ──────────────────────────────────\n')
        f.write(sample + '\n\n')

print(f'Samples saved → {samples_path}')

## Cell 13 — Save Metrics (used in Part 2 comparison)

In [ ]:
baseline_metrics = {
    'model_name'        : 'Baseline LSTM',
    'final_train_loss'  : round(final_train_loss, 4),
    'final_val_loss'    : round(final_val_loss,   4),
    'best_val_loss'     : round(best_val_loss,    4),
    'train_perplexity'  : round(train_perplexity, 2),
    'val_perplexity'    : round(val_perplexity,   2),
    'best_perplexity'   : round(best_perplexity,  2),
    'training_time_s'   : round(total_train_time, 1),
    'total_params'      : total_params,
    'epochs_run'        : len(val_losses),
    'best_epoch'        : best_val_epoch,
    'convergence_epoch' : convergence_ep,
}

metrics_path = os.path.join(OUT_DIR, 'baseline_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print('BASELINE METRICS')
print('=' * 42)
for k, v in baseline_metrics.items():
    print(f'  {k:<25} {v}')
print(f'\nSaved → {metrics_path}')

## Cell 14 — Hyperparameter Table (for Report)

In [ ]:
print('BASELINE MODEL — HYPERPARAMETER TABLE')
print('=' * 52)
rows = [
    ('Sequence Length',       SEQ_LEN),
    ('Embedding Dimension',   EMBED_DIM),
    ('LSTM Units (Layer 1)',  LSTM_UNITS),
    ('LSTM Units (Layer 2)',  LSTM_UNITS),
    ('Dropout Rate',          DROPOUT_RATE),
    ('Batch Size',            BATCH_SIZE),
    ('Learning Rate',         LEARNING_RATE),
    ('Optimizer',             'Adam'),
    ('Loss Function',         'Cross-Entropy'),
    ('Vocabulary Size',       VOCAB_SIZE),
    ('Trainable Parameters',  f'{total_params:,}'),
]
for name, val in rows:
    print(f'  {name:<30} {val}')
print('=' * 52)
print(f'\nAll outputs saved to → {OUT_DIR}/')
print('Part 1 complete ✓')